master of containers

In [ ]:

Big Picture

This project is a production-style image inference service for one task: detect person objects in images using a D-FINE model.

At a systems level, it is split into 3 logical parts:

API tier
Receives HTTP requests, validates them, decides whether to accept/reject them, pushes jobs into Redis, and returns results or job IDs.

Queue/result backend
Redis is used twice:

as the job queue
as temporary result storage
Worker tier
Pulls jobs from Redis, decodes images, batches them, runs the D-FINE model, and stores results back in Redis.



In [ ]:
backpressure-
4. It applies backpressure
If queue depth is too high, the API rejects the request:

compares depth with settings.queue_hard_limit
returns 429 queue saturated

Backpressure
When overloaded, reject early instead of letting latency explode.

In [ ]:
Micro-batching
Wait a tiny time window to combine multiple jobs into one GPU call.

In [ ]:
If clients supply request_id, they can conceptually correlate/retry using the same ID.
This repo does not fully enforce uniqueness, so clients must manage collisions themselves.

In [ ]:
TTL
Temporary storage with expiration.

In [ ]:
Consumer groups
In Redis streams, multiple workers can coordinate message consumption.

In [ ]:
Postprocessing
Turning raw model tensors into usable API JSON.

In [ ]:
# how detect async works

# detect_async is the endpoint that accepts an image, puts it into Redis,
# and immediately returns a job_id instead of waiting for inference to finish.

1)

What it does step by step
1. FastAPI parses the request body
The endpoint expects a DetectRequest object:

image_base64
optional request_id
optional timeout_ms

If the JSON is invalid or missing fields, FastAPI returns 422.


2)It starts request tracking
At the top of the function:

records start time
increments API_INFLIGHT

3)

It checks Redis queue depth
Before accepting more work, it asks Redis how many jobs are already waiting:

depth = queue.queue_depth()

If Redis is unavailable:

request is counted as redis_unavailable
API returns 503



4. It applies backpressure
If queue depth is too high, the API rejects the request:

compares depth with settings.queue_hard_limit
returns 429 queue saturated


5. It creates a job_id
The endpoint uses:

req.request_id if client provided one
otherwise generates a UUID


6. It enqueues the job into Redis
The API pushes this payload to Redis:

{
    "job_id": job_id,
    "image_base64": req.image_base64,
    "submitted_at": time.time(),
}


# Depending on config, Redis stores it as:

# a list item using RPUSH, or
# a stream entry using XADD


7. It immediately returns queued
If enqueue succeeds, API responds:

{
  "job_id": "...",
  "status": "queued"
}

main.py (line 74)
This is why it is called async:
the request does not wait for model inference.

8. It finishes metrics bookkeeping
In the finally block, it:

records request latency
decrements API_INFLIGHT






In [ ]:
What happens after the API returns

After the client already got job_id, the worker does the real work.


runner.py
It

dequeues jobs from Redis
batches them
decodes base64 image to imgbytes
runs D-FINE inference
stores result in Redis with TTL






Then the client polls:

GET /v1/jobs/{job_id}
Code:

main.py (line 81)
If result not ready:

{"job_id":"...", "status":"pending"}
If ready:

{
  "job_id":"...",
  "status":"completed",
  "detections":[...]
}

In [ ]:
# prometheus

Instead of defining metrics separately in API and worker code, the project keeps them
in one shared place so both sides can import the same metric objects.

In [ ]:
#batchers 

batcher.py
It does not perform batching itself.
It only makes the decision: "flush now" or "keep waiting".

The worker uses it here:


GPU inference is usually more efficient with batches, but waiting too long hurts latency.

So the system needs a policy like:

if enough jobs arrive quickly, batch them together
if not enough jobs arrive, don’t wait forever